In [2]:
import os
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
load_dotenv()

api_key = os.getenv("OPENROUTER_API_KEY")
if not api_key:
    raise ValueError("Не найден OPENROUTER_API_KEY. Установи переменную окружения.")

llm = ChatOpenAI(
    model="qwen3.8-27b",                      
    api_key=api_key,
    base_url="https://openrouter.ai/api/v1",
    temperature=0,                                # для стабильного JSON в части 2
    extra_body={"reasoning": {"enabled": False}}, # выключаем размышление 
)

# Проверяем, что соединение работает
response = llm.invoke("Скажи одно слово: работает?")
print("Тип ответа:", type(response))
print("Содержимое:", response.content)
print("usage_metadata:", response.usage_metadata)

Тип ответа: <class 'langchain_core.messages.ai.AIMessage'>
Содержимое: Да
usage_metadata: {'input_tokens': 20, 'output_tokens': 2, 'total_tokens': 22, 'input_token_details': {'audio': 0, 'cache_read': 0, 'cache_creation': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}


## задание 1.1

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Параметризованный промпт:
#    - system: роль модели
#    - human: переменная query (запрос) и language (язык ответа)
prompt = ChatPromptTemplate.from_messages([
    ("system",
     "Ты — ассистент интернет-магазина. Твоя задача — определить категорию товара "
     "по запросу пользователя и кратко ответить. Отвечай на языке: {language}."),
    ("human", "{query}"),
])

# LCEL-цепочка: промпт → LLM → парсер строки
chain = prompt | llm | StrOutputParser()

Сравнение: прямой вызов LLM vs вызов цепочки


In [4]:
direct_response = llm.invoke("Определи категорию товара: ноутбук до 70 тысяч рублей")
chain_response = chain.invoke({
    "query": "Определи категорию товара: ноутбук до 70 тысяч рублей",
    "language": "русский",
})

print("Прямой вызов LLM:")
print("  тип:", type(direct_response))
print("  содержимое:", direct_response.content[:100], "...")
print()
print("Вызов цепочки:")
print("  тип:", type(chain_response))
print("  содержимое:", chain_response[:100], "...")

Прямой вызов LLM:
  тип: <class 'langchain_core.messages.ai.AIMessage'>
  содержимое: Категория товара: **Ноутбуки** (или **Компьютеры и ноутбуки**). ...

Вызов цепочки:
  тип: <class 'langchain_core.messages.base.TextAccessor'>
  содержимое: Ноутбуки ...


In [ ]:
print(type(chain_response).__mro__)   # посмотреть цепочку классов
print(isinstance(chain_response, str))  

(<class 'langchain_core.messages.base.TextAccessor'>, <class 'str'>, <class 'object'>)
True


In [5]:
queries = [
    {"query": "Найди ноутбуки дешевле 70 000 рублей", "language": "русский"},
    {"query": "Покажи самые дорогие смартфоны", "language": "русский"},
    {"query": "Хочу посмотреть планшеты", "language": "русский"},
]

batch_results = chain.batch(queries)

for q, r in zip(queries, batch_results):
    print(f"Запрос: {q['query']}")
    print(f"Ответ:  {r}\n")

Запрос: Найди ноутбуки дешевле 70 000 рублей
Ответ:  Ноутбуки

Запрос: Покажи самые дорогие смартфоны
Ответ:  Смартфоны

Запрос: Хочу посмотреть планшеты
Ответ:  Планшеты

